In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
import csv
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, Bidirectional, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [3]:
# df1 = pd.read_csv('linkedin_jobs_20260418_215155.csv')
# df2 = pd.read_csv('linkedin_jobs_20260416_223428.csv')
# df3 = pd.read_csv('jobstreet_jobs_20260420_205621.csv')
# df = pd.concat([df1, df2, df3])

df = pd.read_csv('linkedin_jobs_final_20260506_115218.csv',
                 engine='python',
                 quoting=csv.QUOTE_MINIMAL,
                 on_bad_lines='skip')

# Pas scraping / sebelum to_csv, strip newline di job_description
df['job_description'] = df['job_description'].str.replace('\n', ' ').str.replace('\r', ' ')

# Lalu simpan dengan quoting yang proper
df.to_csv('output.csv', index=False, quoting=csv.QUOTE_ALL)

rows, cols = df.shape

# print(f"Rows : {rows}")
# print(f"Cols : {cols}")
df.info()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'linkedin_jobs_final_20260506_115218.csv'

In [70]:
print(f"Jumlah unique role: {df['search_role'].nunique()}")
print()
print(df['search_role'].value_counts())

Jumlah unique role: 11

search_role
Web Developer           812
Data Scientist          793
IT                      770
Software Engineer       561
Software Developer      353
Developer               300
Full stack Developer    259
Data Engineer           124
Software                 89
Backend Developer        36
Progammer                14
Name: count, dtype: int64


In [64]:
df = df.drop_duplicates()
df = df.drop(columns='salary', errors='ignore')

print(f"total duplicated columns : {df.duplicated().sum()}")
print(f"total null column : {df.isnull().sum()}")

total duplicated columns : 0
total null column : id                   0
title                0
standardized_role    0
company              0
location             0
city                 0
work_type            0
seniority            0
jd_length            0
job_description      0
extracted_skills     0
skills_count         0
job_url              0
search_role          0
scraped_at           0
dtype: int64


In [65]:
df["job_description"] = df["job_description"].fillna("").astype(str)
df["extracted_skills"] = df["extracted_skills"].fillna("").astype(str)

df["text_input"] = df["job_description"] + " " + df["extracted_skills"]

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])

num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")

print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

max_vocab_length = 20000
max_sequence_length = 512

vectorizer = TextVectorization(
    max_tokens=max_vocab_length,
    output_mode='int',
    output_sequence_length=max_sequence_length
)ƒ

vectorizer.adapt(X_train.to_numpy())

print("\nShape X_train:", X_train.shape)
print("Contoh kalimat pertama setelah di-vektorisasi (angka):")
print(vectorizer(X_train.iloc[0:1]).numpy())

Total role unik: 11
Mapping kelas:
{'Backend Developer': 0, 'Data Engineer': 1, 'Data Scientist': 2, 'Developer': 3, 'Full stack Developer': 4, 'IT': 5, 'Progammer': 6, 'Software': 7, 'Software Developer': 8, 'Software Engineer': 9, 'Web Developer': 10}

Shape X_train: (3288,)
Contoh kalimat pertama setelah di-vektorisasi (angka):
[[   20     4    18  1723    15  1581   486   295  5116  1172    73     7
     14   534     6    65  4764  1723  4529  1723  4217  1723  5140     2
   1723  3302    13  1531     3  1175  1789     2   573    17   318  1192
      4   127   461     6    86   515     5    14   889    59     6   480
     13    47  4501   319   295  1214   133   322     2   565   131   265
      3  3333   123   263  1970    13   371    69   103   226    16  3504
      2  1566     7  3883  1630   122    19   562     3  4851     2  1590
   1581    65   811    18   183    44     2    75     4    49    25    27
    147  2348     6   515     2   737     6   322    22   220     7    49
 

In [66]:
print(type(X_train), X_train.dtype)
print(X_train.head(3))
print(X_train.astype(str).to_numpy().dtype)

<class 'pandas.core.series.Series'> object
2070    About the job  Mekari is Indonesia's no. 1 Sof...
359     About the job  Prudential Syariah’s purpose is...
171     About the job  Introduction  If you enjoy turn...
Name: text_input, dtype: object
object


In [67]:
import numpy as np

class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=3):
        super(CustomEarlyStopping, self).__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Overfitting terdeteksi. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

In [68]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=128)(x)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier_LSTM")

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMulai proses training...")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[CustomEarlyStopping(patience=3)]
)



Mulai proses training...
Epoch 1/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 13s 89ms/step - accuracy: 0.0961 - loss: 2.4019 - val_accuracy: 0.1798 - val_loss: 2.3907
Epoch 2/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 12s 105ms/step - accuracy: 0.1344 - loss: 2.3837 - val_accuracy: 0.1762 - val_loss: 2.2137
Epoch 3/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 18s 79ms/step - accuracy: 0.2251 - loss: 2.2043 - val_accuracy: 0.1896 - val_loss: 2.1475
Epoch 4/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 10s 96ms/step - accuracy: 0.1913 - loss: 2.0652 - val_accuracy: 0.2199 - val_loss: 2.0807
Epoch 5/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 9s 84ms/step - accuracy: 0.2655 - loss: 1.9364 - val_accuracy: 0.2382 - val_loss: 2.0385
Epoch 6/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 10s 84ms/step - accuracy: 0.2737 - loss: 1.7729 - val_accuracy: 0.2588 - val_loss: 2.0191
Epoch 7/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - accuracy: 0.2789 - loss: 1.7241 - val_accuracy: 0.2685 - val_loss: 2.0583
Epoch 8/15
103/103 ━━━━━━━━━━━━━━━━━━━━ 9s 83ms/step - accuracy:

In [69]:
model_path = 'job_role_model.keras'
model.save(model_path)
print(f"[INFO] Model berhasil diekspor ke: {model_path}")

loaded_model = tf.keras.models.load_model(model_path)

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)

    pred_probs = loaded_model.predict(input_tensor, verbose=0)

    pred_class_idx = np.argmax(pred_probs)

    predicted_role = encoder.inverse_transform([pred_class_idx])[0]

    return predicted_role

# 3. Test fungsi inference
skill_input = "saya punya pengalaman bikin antarmuka web pakai reactjs, next.js, tailwind, dan biasa pakai linux ubuntu"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil diekspor ke: job_role_model.keras



--- Hasil Inference ---
Skill User : saya punya pengalaman bikin antarmuka web pakai reactjs, next.js, tailwind, dan biasa pakai linux ubuntu
Cocok untuk: Backend Developer
